In [0]:
# ── Silver: fact_games ───────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DateType

bronze_games = spark.table("football_catalog.bronze.games")
target_table = "football_catalog.silver.fact_games"

print("--- Processing: fact_games ---")

# Clean and cast the core columns from the bronze games table
silver_games = (bronze_games
    .select(
        F.col("game_id").cast(IntegerType()),
        F.col("competition_id"),
        F.col("season").cast(IntegerType()),
        F.col("round"),
        F.to_date(F.col("date"), "yyyy-MM-dd").alias("match_date"),
        F.col("home_club_id").cast(IntegerType()),
        F.col("away_club_id").cast(IntegerType()),
        F.col("home_club_goals").cast(IntegerType()),
        F.col("away_club_goals").cast(IntegerType()),
        F.col("attendance").cast(IntegerType()),
        F.col("stadium")
    )
    # Filter out any rows where game_id is null (data quality check)
    .filter(F.col("game_id").isNotNull())
    # Ensure there are no duplicate games
    .dropDuplicates(["game_id"])
)

# Write to Silver layer
(silver_games.write
 .format("delta")
 .mode("overwrite") 
 .option("overwriteSchema", "true")
 .saveAsTable(target_table))

print(f"SUCCESS: Created {target_table}")
print(f"Total rows: {spark.table(target_table).count():,}")